# Assignment 2: Gradio dashboard

You're using a notebook for this stage. If you'd rather use a script,
write `app.py` instead and delete this file. `main.py` refuses to run if
it finds both `app.py` and `app.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py app`, this notebook must build and launch a Gradio app
(a `demo` that calls `.launch()`) using only what your training stage already
produced. It must never train anything itself. Load the committed weights in
`weights/`, never whatever the training run left in `runs/`.

At minimum, the dashboard must let you:

- Upload an image and see the model's predictions at the current threshold.
- Move the two edges of your escalation band and watch the confusion matrix,
  the expected cost per 1,000 units, and the share of cases escalated to a
  human all change together.
- See the cost-against-threshold curve with your chosen operating point marked
  on it.
- See per-class performance, not just the overall number.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.

In [ ]:
import json
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
from matplotlib.figure import Figure
from PIL import ImageDraw
from ultralytics import YOLO

import costing

ROOT = Path.cwd()
RESULTS = ROOT / "results"

model = YOLO(str(ROOT / "weights" / "best.pt"))
metrics = json.load(open(RESULTS / "metrics.json"))
sweep = json.load(open(RESULTS / "cost_sweep.json"))
scores = json.load(open(RESULTS / "scores.json"))

A = sweep["assumptions"]            # the assumptions used in the report
CHOSEN = sweep["chosen"]            # operating point chosen by the sweep
CRACKED_ID = [k for k, v in model.names.items() if v == "cracked"][0]
SPLITS = {
    name: costing.split_scores(np.array(scores[name]["score"]), np.array(scores[name]["cracked"]))
    for name in ("test", "valid")
}
DECISIONS = ["Pass", "Human review", "Fail (remove egg)"]


def refresh(split, lo, hi, base_rate):
    """Recompute cost, review load, confusion matrix and cost curve."""
    note = ""
    if hi < lo:
        note = "\n\n*Upper threshold was below the lower one, so the upper threshold is set equal to the lower.*"
        hi = lo
    pos, neg = SPLITS[split]
    B = {**A, "base_rate": base_rate}
    r = costing.expected_cost(pos, neg, lo, hi, B)
    per_day, staff = costing.staffing(r, B)
    nothing = costing.expected_cost(pos, neg, 1.01, 1.01, B)["cost_per_1000"]
    d = r["decisions_per_1000"]
    keys = ("pass", "review", "fail")

    summary = (
        f"### Expected cost: €{r['cost_per_1000']:.2f} per 1,000 eggs "
        f"(doing nothing: €{nothing:.2f})\n"
        f"- **Sent to a person:** {r['review_share']:.1%} of eggs, about {per_day:,.0f} checks per day, "
        f"{staff:.1f} reviewer(s) needed\n"
        f"- **After review:** {r['final_missed_cracked_per_1000']:.2f} cracked eggs missed and "
        f"{r['final_good_removed_per_1000']:.2f} good eggs removed per 1,000\n"
        f"- Assumed share of cracked eggs: {base_rate:.1%}. Scores come from the {split} split "
        f"({len(pos)} cracked, {len(neg)} not cracked)." + note
    )
    per1000 = pd.DataFrame({
        "Model decision": DECISIONS,
        "Truly cracked": [round(d[k]["cracked"], 1) for k in keys],
        "Not cracked": [round(d[k]["not_cracked"], 1) for k in keys],
    })
    raw = pd.DataFrame({
        "Model decision": DECISIONS,
        "Truly cracked": [int((pos < lo).sum()), int(((pos >= lo) & (pos < hi)).sum()), int((pos >= hi).sum())],
        "Not cracked": [int((neg < lo).sum()), int(((neg >= lo) & (neg < hi)).sum()), int((neg >= hi).sum())],
    })

    grid = costing.GRID[costing.GRID <= 1]
    curve = [costing.expected_cost(pos, neg, t, t, B)["cost_per_1000"] for t in grid]
    t_star = CHOSEN["single_threshold"]
    fig = Figure(figsize=(7, 4))
    ax = fig.subplots()
    ax.plot(grid, curve, label=f"{split} split, single threshold")
    ax.axvline(t_star, color="k", ls=":", label=f"chosen operating point ({t_star:.2f})")
    ax.scatter([t_star], [costing.expected_cost(pos, neg, t_star, t_star, B)["cost_per_1000"]], color="k", zorder=3)
    if hi > lo:
        ax.axvspan(lo, hi, color="orange", alpha=0.25, label="current review band")
    else:
        ax.axvline(lo, color="tab:red", lw=1, label=f"current threshold ({lo:.2f})")
    ax.set_yscale("log")
    ax.set_xlabel("confidence threshold on the 'cracked' score")
    ax.set_ylabel("expected cost per 1,000 eggs (EUR, log scale)")
    ax.set_title("Cost against threshold")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return summary, per1000, raw, fig


def predict(img, lo, hi):
    """Run the model on one uploaded image and apply the two thresholds."""
    if img is None:
        return None, "Upload an image to see predictions."
    hi = max(hi, lo)
    res = model.predict(img, conf=0.001, iou=0.6, max_det=50, verbose=False, device="cpu")[0]
    cls = [int(c) for c in res.boxes.cls.tolist()]
    conf = res.boxes.conf.tolist()
    boxes = res.boxes.xyxy.tolist()
    score = max([c for k, c in zip(cls, conf) if k == CRACKED_ID], default=0.0)
    if score < lo:
        decision = "PASS"
    elif score >= hi:
        decision = "FAIL: remove the egg"
    else:
        decision = "HUMAN REVIEW"
    show = max(lo, 0.01)
    out = img.convert("RGB").copy()
    draw = ImageDraw.Draw(out)
    for k, c, b in zip(cls, conf, boxes):
        if c >= show:
            color = "red" if k == CRACKED_ID else "blue"
            draw.rectangle(b, outline=color, width=4)
            draw.text((b[0] + 4, b[1] + 4), f"{model.names[k]} {c:.2f}", fill=color)
    text = (f"### Decision: {decision}\nCracked score: **{score:.3f}** "
            f"(pass below {lo:.2f}, fail at or above {hi:.2f}).\n\n"
            f"Boxes shown have confidence of at least {show:.2f}. The rule assumes one egg per image.")
    return out, text


# ---- static per-class results (from metrics.json) ----
fine = metrics["finetuned"]
per_class_df = pd.DataFrame([
    {"Class": k, "Precision": round(v["precision"], 3), "Recall": round(v["recall"], 3),
     "AP50": round(v["ap50"], 3), "mAP50-95": round(v["ap50_95"], 3)}
    for k, v in fine["per_class"].items()
])
models_df = pd.DataFrame([
    {"Model": "Finetuned yolo11n", "mAP50": round(fine["map50"], 3), "mAP50-95": round(fine["map50_95"], 3)},
    {"Model": "COCO yolo11n, zero-shot", "mAP50": round(metrics["coco_zero_shot"]["map50"], 3),
     "mAP50-95": round(metrics["coco_zero_shot"]["map50_95"], 3)},
    {"Model": "Predict nothing", "mAP50": 0.0, "mAP50-95": 0.0},
])
cm = fine["confusion_matrix"]
cm_df = pd.DataFrame(cm["matrix"], columns=[f"true: {l}" for l in cm["labels"]])
cm_df.insert(0, "", [f"predicted: {l}" for l in cm["labels"]])

# ---- layout ----
with gr.Blocks(title="Cracked-egg inspector") as demo:
    gr.Markdown(
        "# Cracked-egg inspector\n"
        "Eggs are scored by the highest 'cracked' confidence. Below the lower threshold they pass, "
        "at or above the upper threshold they are removed, and in between a person checks them. "
        "Positive class: **cracked**."
    )
    with gr.Row():
        lo = gr.Slider(0, 1, value=min(CHOSEN["lo"], 1), step=0.01, label="Pass below this score")
        hi = gr.Slider(0, 1, value=min(CHOSEN["hi"], 1), step=0.01, label="Fail at or above this score")
    with gr.Row():
        base = gr.Slider(0.005, 0.5, value=A["base_rate"], step=0.005, label="Assumed share of cracked eggs")
        split = gr.Radio(["test", "valid"], value="test", label="Split used for scores")

    with gr.Tab("Cost and review"):
        summary = gr.Markdown()
        with gr.Row():
            t_expected = gr.Dataframe(label="Decisions per 1,000 eggs at the assumed base rate")
            t_raw = gr.Dataframe(label="Raw image counts in the selected split")
        plot = gr.Plot(label="Cost against threshold")

    with gr.Tab("Per-class results"):
        gr.Markdown("Test split, positive class **cracked**. Precision and recall are at the F1-optimal confidence.")
        gr.Dataframe(value=per_class_df, label="Per class")
        gr.Dataframe(value=models_df, label="Finetuned model against baselines")
        gr.Dataframe(value=cm_df, label="Ultralytics confusion matrix (rows predicted, columns true; includes low-confidence boxes)")

    with gr.Tab("Try an image"):
        with gr.Row():
            img = gr.Image(type="pil", label="Upload an egg image")
            shown = gr.Image(label="Predictions")
        verdict = gr.Markdown()

    controls = [split, lo, hi, base]
    outputs = [summary, t_expected, t_raw, plot]
    for c in controls:
        c.change(refresh, controls, outputs)
    demo.load(refresh, controls, outputs)
    for c in (img, lo, hi):
        c.change(predict, [img, lo, hi], [shown, verdict])

demo.launch()